In [1]:
import sqlite3
import pandas as pd
import json
from pathlib import Path

In [2]:
#connect to db

db_path = Path("/Users/chowingchan/Desktop/Project/AI-Analytics-Copilot/Competitive-Intelligence-Internal-Analytics-System/data/database/thelook_ecommerce.db")
conn = sqlite3.connect(db_path)

In [3]:
#check if we successfully connect to SQLite database
pd.read_sql("""
SELECT name, type
FROM sqlite_master
WHERE type IN ('table', 'view')
ORDER BY type, name;
""", conn)

,name,type
0,distribution_centers,table
1,events,table
2,inventory_items,table
3,mart_user_segment,table
4,order_items,table
5,orders,table
6,products,table
7,users,table
8,mart_daily_sales,view
9,mart_order_summary,view


## Question Categories

| Category          | Example           |
| ----------------- | ----------------- |
| revenue KPI       | total revenue     |
| user KPI          | active users      |
| product analysis  | top categories    |
| ranking           | top 5 products    |
| segmentation      | high value users  |
| time analysis     | last 30 days      |
| return analysis   | return rate       |
| delivery analysis | avg delivery days |


In [51]:
#Compare function
def evaluate_sql_result(expected_df, generated_df):
    if expected_df.shape != generated_df.shape:
        return {
            "passed": False,
            "reason": "Shape mismatch",
            "expected_shape": expected_df.shape,
            "generated_shape": generated_df.shape
        }

    if list(expected_df.columns) != list(generated_df.columns):
        return {
            "passed": False,
            "reason": "Column mismatch",
            "expected_columns": list(expected_df.columns),
            "generated_columns": list(generated_df.columns)
        }

    if expected_df.equals(generated_df):
        return {
            "passed": True,
            "reason": "Result matched"
        }

    return {
        "passed": False,
        "reason": "Result values mismatch",
        "expected_result": expected_df.to_dict(orient="records"),
        "generated_result": generated_df.to_dict(orient="records")
    }

In [52]:
# Main evaluator 

def evaluate_sql(
    question,
    expected_sql,
    generated_sql,
    conn
):
    try:
        expected_df = pd.read_sql(expected_sql, conn)
        
    except Exception as e:
        return{
            "question": question,
            "passed": False,
            "stage": "expected_sql_execution",
            "error": str(e)
        }
    
    try:
        generated_df = pd.read_sql(generated_sql, conn)
        
    except Exception as e:
        return{
            "question": question,
            "passed": False,
            "stage": "generated_sql_execution",
            "error":str(e)
        }
    
    comparison_result = evaluate_sql_result(
        expected_df,
        generated_df
    )
    
    comparison_result["question"] = question
    return comparison_result
        

In [53]:
# test revenue

question_1 = "What is the total revenue in 2021"

expected_sql_1 = """
SELECT 
    ROUND(SUM(order_revenue),2) AS total_revenue 
FROM mart_order_summary
WHERE strftime('%Y', order_created_at)='2021' AND order_returned_at IS NULL;
"""
expected_result = pd.read_sql(expected_sql_1, conn)

generated_sql_1 = """
SELECT 
    ROUND(SUM(order_revenue),2) AS total_revenue 
FROM mart_order_summary
WHERE strftime('%Y', order_created_at)='2021';
"""
generated_result = pd.read_sql(generated_sql_1, conn)

generated_sql_1_correct = """
SELECT
    ROUND(SUM(order_revenue), 2) AS total_revenue
FROM mart_order_summary
WHERE strftime('%Y', order_created_at)='2021'
  AND order_returned_at IS NULL;
"""

In [62]:
# Top 5 product categories by revenue

question_2 = "What are the top 5 product categories by revenue in 202101?"

expected_sql_2 = """
SELECT 
    category,
    ROUND(SUM(gross_sales),2) AS total_revenue 
FROM mart_product_sales
WHERE strftime('%Y%m', order_date) = '202101'
GROUP BY category
ORDER BY total_revenue DESC
LIMIT 5;
"""
expected_result = pd.read_sql(expected_sql_2, conn)

generated_sql_2_correct = """
SELECT 
    category,
    ROUND(SUM(gross_sales),2) AS total_revenue 
FROM mart_product_sales
WHERE strftime('%Y%m', order_date) = '202101'
GROUP BY category
ORDER BY total_revenue DESC
LIMIT 5;
"""

generated_sql_2_wrong = """
SELECT 
    category,
    ROUND(SUM(gross_sales),2) AS total_revenue 
FROM mart_product_sales
WHERE strftime('%Y%m', order_date) = '202101'
GROUP BY category
ORDER BY total_revenue DESC;
"""

In [84]:
#How many active users purchased in the last 90 days
question_3 = "How many active users purchased in the last 90 days?"

expected_sql_3 = """
SELECT 
    COUNT(DISTINCT user_id) AS active_users 
FROM mart_user_segment
WHERE customer_segment != "Inactive Users" AND recency <= '90';
"""
expected_result = pd.read_sql(expected_sql_3, conn)

generated_sql_3_correct = """
SELECT 
    COUNT(DISTINCT user_id) AS active_users 
FROM mart_user_segment
WHERE customer_segment != "Inactive Users" AND recency <= '90';
"""

generated_sql_3_wrong = """
SELECT 
    COUNT(DISTINCT user_id) AS active_users 
FROM mart_user_segment
WHERE customer_segment != "Inactive Users";
"""

In [90]:
question_4 = "What is the average order value in 2021? "

expected_sql_4 = """
    SELECT SUM(gross_sales)/SUM(total_orders)
    FROM mart_product_sales
    WHERE strftime('%Y', order_date) = '2021'
"""
expected_result = pd.read_sql(expected_sql_4, conn)

generated_sql_4_correct = """
    SELECT SUM(gross_sales)/SUM(total_orders)
    FROM mart_product_sales
    WHERE strftime('%Y', order_date) = '2021'
"""

generated_sql_4_wrong = """
    SELECT SUM(gross_sales)/SUM(units_sold)
    FROM mart_product_sales
"""

In [96]:
question_5 = "What is the return rate in 2022?"

expected_sql_5 = """
    SELECT (SUM(returned_units) * 1.0) / SUM(units_sold) AS return_rate
    FROM mart_product_sales
    WHERE strftime('%Y', order_date) = '2022';
"""
expected_result = pd.read_sql(expected_sql_5, conn)

generated_sql_5_correct = """
    SELECT (SUM(returned_units) * 1.0) / SUM(units_sold) AS return_rate
    FROM mart_product_sales
    WHERE strftime('%Y', order_date) = '2022';
"""

generated_sql_5_wrong = """
    SELECT SUM(returned_units) AS return_rate
    FROM mart_product_sales
    WHERE strftime('%Y', order_date) = '2022';
"""

In [97]:
test_cases = [
    {
        "question": question_1,
        "expected_sql": expected_sql_1,
        "generated_sql": generated_sql_1_correct
    },
    {
        "question": question_2,
        "expected_sql": expected_sql_2,
        "generated_sql": generated_sql_2_wrong
    },
    {
        "question": question_3,
        "expected_sql": expected_sql_3,
        "generated_sql": generated_sql_3_wrong
    },
    {
        "question": question_4,
        "expected_sql": expected_sql_4,
        "generated_sql": generated_sql_4_correct
    },
    {
        "question": question_5,
        "expected_sql": expected_sql_5,
        "generated_sql": generated_sql_5_wrong
    }
]

In [ ]:
with open("../data/evaluation/test_cases.json", "r") as f:
    loaded_test_cases = json.load(f)

loaded_test_cases

In [101]:
evaluation_results = []

for case in loaded_test_cases:
    result = evaluate_sql(
    case["question"],
    case["expected_sql"],
    case["generated_sql"],
    conn
    )
    evaluation_results.append(result)
    
pd.DataFrame(evaluation_results)

,passed,reason,question,expected_shape,generated_shape,expected_result,generated_result
0,True,Result matched,What is the total revenue in 2021,NaN,NaN,NaN,NaN
1,False,Shape mismatch,What are the top 5 product categories by reven...,"(5, 2)","(26, 2)",NaN,NaN
2,False,Result values mismatch,How many active users purchased in the last 90...,NaN,NaN,[{'active_users': 24983}],[{'active_users': 71408}]
3,True,Result matched,What is the average order value in 2021?,NaN,NaN,NaN,NaN
4,False,Result values mismatch,What is the return rate in 2022?,NaN,NaN,[{'return_rate': 0.39530632702399904}],[{'return_rate': 6704}]


In [73]:
# Summary report
evaluation_summary = pd.DataFrame(evaluation_results)
total_cases = len(evaluation_summary)
passed_cases = evaluation_summary["passed"].sum()
failed_cases = total_cases - passed_cases
pass_rate = evaluation_summary["passed"].mean()

summary_report = {
    "total_case": int(total_cases),
    "passed_cases": int(passed_cases),
    "failed_cases": int(failed_cases),
    "pass_rate": float(round(pass_rate, 2))
}

summary_report

{'total_case': 2, 'passed_cases': 1, 'failed_cases': 1, 'pass_rate': 0.5}

In [75]:
# save as csv

evaluation_summary.to_csv(
    "../outputs/sql_evaluation_result.csv",
    index=False
)

with open("../outputs/sql_evaluation_summary.json", "w") as f:
    json.dump(summary_report, f, indent=2)

In [99]:
# Create test_cases json file

Path("../data/evaluation").mkdir(parents=True, exist_ok=True)

with open("../data/evaluation/test_cases.json", "w") as f:
    json.dump(test_cases, f, indent=2)